In [1]:
import sys

print(sys.executable)

c:\Users\saads\from-training-to-deployment\.venv\Scripts\python.exe


# 01 — Data Exploration & Preprocessing

## Projet : From Training to Deployment

### Objectif
Préparer un dataset de critiques de films en français pour entraîner un modèle de Deep Learning capable de classifier les sentiments.

### Dataset
Allociné — avis positifs et négatifs.

### Étapes de travail
1. Chargement du dataset
2. Exploration des données (EDA)
3. Analyse de la qualité des données
4. Nettoyage et preprocessing
5. Tokenization avec DistilBERT
6. Préparation des données pour l'entraînement

## 1. Chargement du dataset Allociné

Nous utilisons le dataset Allociné disponible sur Hugging Face.

Il contient des critiques de films en français associées à des étiquettes de sentiment.

In [2]:
# Importer la fonction de chargement
from datasets import load_dataset

# Charger le dataset Allociné
dataset = load_dataset("tblard/allocine")

# Afficher sa structure
print(dataset)

c:\Users\saads\from-training-to-deployment\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\saads\from-training-to-deployment\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\saads\.cache\huggingface\hub\datasets--tblard--allocine. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In 

DatasetDict({
    train: Dataset({
        features: ['review', 'label'],
        num_rows: 160000
    })
    validation: Dataset({
        features: ['review', 'label'],
        num_rows: 20000
    })
    test: Dataset({
        features: ['review', 'label'],
        num_rows: 20000
    })
})


In [4]:
# Afficher les ensembles disponibles
print(dataset)

# Afficher les colonnes
print(dataset["train"].column_names)

# Afficher le premier commentaire
print(dataset["train"][0])

# Afficher le nombre de lignes
print("Nombre de lignes :", len(dataset["train"]))

DatasetDict({
    train: Dataset({
        features: ['review', 'label'],
        num_rows: 160000
    })
    validation: Dataset({
        features: ['review', 'label'],
        num_rows: 20000
    })
    test: Dataset({
        features: ['review', 'label'],
        num_rows: 20000
    })
})
['review', 'label']
{'review': 'Si vous cherchez du cinéma abrutissant à tous les étages,n\'ayant aucune peur du cliché en castagnettes et moralement douteux,"From Paris with love" est fait pour vous.Toutes les productions Besson,via sa filière EuropaCorp ont de quoi faire naître la moquerie.Paris y est encore une fois montrée comme une capitale exotique,mais attention si l\'on se dirige vers la banlieue,on y trouve tout plein d\'intégristes musulmans prêts à faire sauter le caisson d\'une ambassadrice américaine.Nauséeux.Alors on se dit qu\'on va au moins pouvoir apprécier la déconnade d\'un classique buddy-movie avec le jeune agent aux dents longues obligé de faire équipe avec un vieux lou comp

## 2. Première exploration des données

### Analyse de la distribution des sentiments

Nous allons examiner la répartition des classes positives et négatives afin de vérifier si le dataset est équilibré.

In [5]:
# Compter le nombre de critiques pour chaque label
label_counts = dataset["train"].to_pandas()["label"].value_counts().sort_index()

# Afficher les résultats
print(label_counts)

# Afficher les proportions en pourcentage
print("\nRépartition en pourcentage :")
print(label_counts / len(dataset["train"]) * 100)

label
0    79413
1    80587
Name: count, dtype: int64

Répartition en pourcentage :
label
0    49.633125
1    50.366875
Name: count, dtype: float64


## 3. Analyse de la qualité des données

Nous vérifions la présence de valeurs manquantes, de commentaires vides et de doublons dans le jeu d'entraînement.

In [6]:
import pandas as pd

# Convertir le jeu d'entraînement en DataFrame
df_train = dataset["train"].to_pandas()

# 1. Valeurs manquantes
print("Valeurs manquantes :")
print(df_train.isnull().sum())

# 2. Commentaires vides
empty_reviews = df_train["review"].fillna("").str.strip().eq("").sum()
print("\nCommentaires vides :", empty_reviews)

# 3. Commentaires dupliqués
duplicate_reviews = df_train["review"].duplicated().sum()
print("Commentaires dupliqués :", duplicate_reviews)

# 4. Dimensions du DataFrame
print("\nDimensions :", df_train.shape)

Valeurs manquantes :
review    0
label     0
dtype: int64

Commentaires vides : 0
Commentaires dupliqués : 539

Dimensions : (160000, 2)


In [10]:
# Détecter les commentaires dupliqués
duplicates = df_train[df_train["review"].duplicated(keep=False)]

print("Nombre de lignes contenant des commentaires dupliqués :",
      len(duplicates))

print("Nombre de doublons supplémentaires :",
      df_train["review"].duplicated().sum())

Nombre de lignes contenant des commentaires dupliqués : 813
Nombre de doublons supplémentaires : 539


In [7]:
# Récupérer les commentaires dupliqués
duplicates = df_train[
    df_train["review"].duplicated(keep=False)
].sort_values("review")

# Afficher les 10 premiers groupes de doublons
print(duplicates.head(20).to_string(index=False))

# Vérifier les doublons ayant des labels différents
label_conflicts = (
    duplicates.groupby("review")["label"]
    .nunique()
)

print("\nNombre de textes associés à plusieurs labels :",
      (label_conflicts > 1).sum())

In [8]:
# Identifier les textes associés à plusieurs labels
conflict_texts = label_conflicts[label_conflicts > 1].index

# Afficher les conflits
for i, review in enumerate(conflict_texts, start=1):

    print(f"\n--- Conflit {i} ---")
    print("Commentaire :", review[:500])

    print(
        "Labels associés :",
        df_train.loc[
            df_train["review"] == review, "label"
        ].unique()
    )


--- Conflit 1 ---
Commentaire : La Wanda du film tient du spectre et de la femme fatale. L'oeuvre baigne dans un romantisme noir comme un deuil qui ne passerait pas. J. Franco donne toute la mesure de son génie de metteur en scène et le résultat est littéralement hypnotique. Bel exemple de la créativité pop de la fin des années 60 (y compris avec la bande son).
Labels associés : [0 1]

--- Conflit 2 ---
Commentaire : Un assez bon film. Évidemment, ce n'est pas une adaptation proprement dite puisque le personnage principal, appelé Louis, fait connaissance avec l'histoire du livre "le Portrait de Dorian Gray" et il décide de faire la même chose. C'est donc une adaptation assez modernisée mais avec beaucoup de références sur le bouquin, surtout dans le personnage d'Henry. Les acteurs sont plutôt bons, le portrait est malheureusement assez mal fait mais on voit tout de même le personnage principal sombré peu à p
Labels associés : [0 1]

--- Conflit 3 ---
Commentaire : bien
Labels associés

In [9]:
# Nombre de lignes concernées par les conflits
conflict_rows = df_train[
    df_train["review"].isin(conflict_texts)
]

print("Textes conflictuels :", len(conflict_texts))
print("Lignes concernées :", len(conflict_rows))

# Nombre de doublons exacts (texte + label)
exact_duplicates = df_train.duplicated(
    subset=["review", "label"]
).sum()

print("Doublons exacts (texte + label) :", exact_duplicates)

Textes conflictuels : 5
Lignes concernées : 15
Doublons exacts (texte + label) : 534


### 4. Nettoyage des commentaires dupliqués

Cette étape consiste à supprimer les commentaires associés à plusieurs labels, car leur sentiment est ambigu.

Les répétitions exactes restantes sont également supprimées afin de conserver une seule occurrence de chaque commentaire.

Le dataset original est conservé et le nettoyage est effectué dans un nouveau DataFrame appelé `df_train_clean`.

In [11]:
# Supprimer les conflits et les commentaires dupliqués
df_train_clean = df_train[
    ~df_train["review"].isin(conflict_texts)
].drop_duplicates(
    subset=["review"],
    keep="first"
).reset_index(drop=True)

print("Nombre de lignes avant :", len(df_train))
print("Nombre de lignes après :", len(df_train_clean))
print("Nombre de lignes supprimées :", len(df_train) - len(df_train_clean))

Nombre de lignes avant : 160000
Nombre de lignes après : 159456
Nombre de lignes supprimées : 544


### 5. Vérification de la distribution des labels après nettoyage

Après le nettoyage, nous vérifions la répartition des commentaires positifs et négatifs.

Cette vérification permet de s'assurer que la suppression des doublons et des conflits n'a pas fortement déséquilibré les deux classes.

In [12]:
print("Distribution avant nettoyage :")
print(
    (df_train["label"].value_counts(normalize=True)
     .sort_index() * 100).round(2)
)

print("\nDistribution après nettoyage :")
print(
    (df_train_clean["label"].value_counts(normalize=True)
     .sort_index() * 100).round(2)
)

Distribution avant nettoyage :
label
0    49.63
1    50.37
Name: proportion, dtype: float64

Distribution après nettoyage :
label
0    49.62
1    50.38
Name: proportion, dtype: float64


### 6. Analyse de la longueur des commentaires

Dans cette étape, nous analysons la longueur des commentaires afin de mieux comprendre la structure du texte.

Nous calculons le nombre de caractères et le nombre de mots de chaque commentaire.

Cette analyse permettra d'identifier les commentaires courts et longs et de préparer le choix de la longueur maximale lors de la tokenisation.

In [13]:
# Calculer la longueur des commentaires
df_train_clean["char_length"] = df_train_clean["review"].str.len()

df_train_clean["word_length"] = (
    df_train_clean["review"].str.split().str.len()
)

# Statistiques descriptives
print(df_train_clean[["char_length", "word_length"]].describe())

         char_length    word_length
count  159456.000000  159456.000000
mean      543.304109      91.428808
std       451.286696      75.384970
min         3.000000       1.000000
25%       197.000000      34.000000
50%       390.000000      66.000000
75%       773.000000     130.000000
max      2000.000000     384.000000


### 7. Visualisation de la longueur des commentaires

Nous utilisons des histogrammes pour visualiser la distribution de la longueur des commentaires.

Ces graphiques permettent d'observer la concentration des commentaires courts et moyens, ainsi que la présence de commentaires particulièrement longs.

In [ ]:
import matplotlib.pyplot as plt

# Distribution du nombre de mots
plt.figure(figsize=(10, 5))
plt.hist(df_train_clean["word_length"], bins=50)
plt.title("Distribution du nombre de mots par commentaire")
plt.xlabel("Nombre de mots")
plt.ylabel("Nombre de commentaires")
plt.show()

# Distribution du nombre de caractères
plt.figure(figsize=(10, 5))
plt.hist(df_train_clean["char_length"], bins=50)
plt.title("Distribution du nombre de caractères par commentaire")
plt.xlabel("Nombre de caractères")
plt.ylabel("Nombre de commentaires")
plt.show()